##Silver Transformation of the drivers bronze table
### 1. Read the table from bronze schema

### Getting the batch id as input parameter

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00.Common/01.Environment-config

In [0]:
%run "../00.Common/03.Helper_Notebook_Silver"

In [0]:
source_name = f"{catalog_name}.{bronze_schema}.drivers"
target_name = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
#import the sql function and filter the df with the batch id
from pyspark.sql import functions as F
drivers_df = (spark.table(source_name).filter(F.col("batch_id")==v_batch_id))
display(drivers_df)

### 2. Drop the unnecessary column (URL) 

In [0]:
drivers_dropped_df = drivers_df.drop("url")

### 3. Rename the columns

In [0]:
drivers_renamed_df = drivers_dropped_df.withColumnsRenamed({"driverId":"driver_id","dateOfBirth":"date_of_birth"})
display(drivers_renamed_df)

### 4. Concatenate the names struct field and creating as a new column and drop the struct field

In [0]:
from pyspark.sql import functions as F
drivers_concat_df = (drivers_renamed_df
                     .withColumn("driver_name",
                                 F.initcap(F.concat_ws(" ", F.col("name.givenName"),F.col("name.familyName"))))
                     .drop("name")
)
display(drivers_concat_df)


In [0]:
# Removing duplicates based on the primary key
drivers_clean_df = drivers_concat_df.dropDuplicates(["driver_id"])
display(drivers_clean_df)

### 5. Transforming the column values 

In [0]:
# Converting the values to initcap format in locality and circuit name columns
drivers_final_df = (drivers_clean_df
                     .withColumn("nationality",F.initcap(F.col("nationality")))
)
display(drivers_final_df)

In [0]:
drivers_final_df.columns

### 6. Writing the final dataframe as table into the silver schema

In [0]:
write_to_silver(
    input_df = drivers_final_df,
    target_table = target_name,
    merge_condition = "t.driver_id=s.driver_id",
    columns_to_update = ['driver_id',
 'date_of_birth',
 'nationality',
 'ingestion_timestamp',
 'SourceFile',
 'batch_id',
 'driver_name']
)

In [0]:
%sql
select * from formula1_incr.silver.drivers;